# LATAM dispute-service analytics

This notebook turns the synthetic hackathon lakehouse into decision-ready baselines for a safer,
faster and more sustainable dispute journey. It reads the treated tables in `lakehouse.silver`
from a local/private-S3 DuckDB file when `ANALYTICS_DUCKDB_URI` is configured, or from MotherDuck
as a compatibility fallback. Both paths are read-only.

**Scope:** synthetic data only. Results describe this dataset—not real customers, national
prevalence, causal effects, fraud, liability, a chargeback outcome or a final Visa code. Public
outputs are aggregate-only.

## 0. Setup and source contract

Set `ANALYTICS_DUCKDB_URI` to the private S3 object or a local `.duckdb` file. For S3, provide a
read-only AWS credential through the normal credential-chain environment variables. If the URI
is absent, set `MOTHERDUCK_TOKEN` for the compatibility source. Secrets are never printed or
embedded in a connection string. Every table section starts by creating a dataframe from its
treated source or an aggregate query.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "analytics").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env")

from analytics.data import (  # noqa: E402
    AnalyticsRepository,
    connect_duckdb_file,
    connect_motherduck,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

duckdb_uri = os.getenv("ANALYTICS_DUCKDB_URI")
if duckdb_uri:
    database = os.getenv("ANALYTICS_DUCKDB_CATALOG", "lakehouse")
    schema = os.getenv("ANALYTICS_DUCKDB_SCHEMA", "silver")
    connection = connect_duckdb_file(duckdb_uri, catalog=database)
else:
    database = os.getenv("MOTHERDUCK_DATABASE", "lakehouse")
    schema = os.getenv("MOTHERDUCK_SCHEMA", "silver")
    connection = connect_motherduck(database=database)
repo = AnalyticsRepository(connection, catalog=database, schema=schema)
available_tables = repo.available_tables()
inventory = repo.data_inventory()
display(inventory)

## 1. Dispute intake and baseline

`CARGO NO RECONOCIDO` and `COBRO INDEBIDO` are synthetic intake labels. They are useful for finding
potential disputes, but they are not final classifications. Linking to `products` restricts the
baseline to credit and debit cards.

In [ ]:
headline_metrics = pd.DataFrame([repo.headline_metrics()])
card_disputes_by_label = repo.card_dispute_breakdown("subcategory")
card_disputes_by_country = repo.card_dispute_breakdown("country")
card_disputes_by_channel = repo.card_dispute_breakdown("channel")
card_disputes_by_status = repo.card_dispute_breakdown("status")
country_experience = repo.card_dispute_country_experience()
card_dispute_trend = repo.card_dispute_monthly_trend()
display(headline_metrics)
display(country_experience)
display(card_dispute_trend.tail(12))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, frame, title in [
    (axes[0, 0], card_disputes_by_label, "Card-linked intake labels"),
    (axes[0, 1], card_disputes_by_country, "Country mix"),
    (axes[1, 0], card_disputes_by_channel, "Reception channel"),
    (axes[1, 1], card_disputes_by_status, "Current status"),
]:
    ax.pie(frame["cases"], labels=frame["category"], autopct="%1.1f%%", startangle=90)
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
trend_plot = card_dispute_trend.set_index("month")
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
trend_plot["cases"].plot(ax=axes[0], color="#2563eb", title="Monthly card-dispute proxy volume")
trend_plot["sla_breach_pct"].plot(ax=axes[1], color="#dc2626", title="Monthly SLA breach rate")
axes[0].set_ylabel("Cases")
axes[1].set_ylabel("Percent")
axes[1].set_xlabel("Month")
plt.tight_layout()
plt.show()

### What the baseline means

- Use the intake-label count to size **demand**, and the card-linked count to size the current
  product scope.
- Compare call-center share, SLA breaches, repeat complainants and resolution time before and after
  the solution. Counts alone are not success metrics.
- Country comparisons are operational diagnostics. They should trigger investigation, not unequal
  treatment.

## 2. Transactions and products

This section creates the transaction/product aggregate dataframe. It helps the team understand
which product and transaction combinations create search volume without exposing customer rows.

In [ ]:
transaction_mix = repo.product_transaction_mix()
display(transaction_mix.head(20))

top_transaction_mix = transaction_mix.nlargest(15, "transactions")
plt.figure(figsize=(11, 6))
sns.barplot(top_transaction_mix, x="transactions", y="product_type", hue="transaction_type")
plt.title("Largest product and transaction-type combinations")
plt.xlabel("Transactions")
plt.ylabel("Product type")
plt.tight_layout()
plt.show()

## 3. Call-center interactions

`was_resolved` is used as an FCR proxy because the source says it indicates resolution on the first
call. Production FCR should also verify that the same issue did not generate another contact inside
a defined window.

In [ ]:
call_outcomes = repo.call_center_outcomes("reason_category")
call_channels = repo.call_center_outcomes("channel")
accent_alignment = repo.accent_alignment()
display(call_outcomes)
display(call_channels)

In [ ]:
def draw_binary_donut(ax, label, value, color="#2563EB"):
    value = min(max(float(value), 0.0), 100.0)
    ax.pie(
        [value, 100.0 - value],
        colors=[color, "#E5E7EB"],
        startangle=90,
        counterclock=False,
        wedgeprops={"width": 0.38, "edgecolor": "white"},
    )
    ax.text(0, 0, f"{value:.1f}%", ha="center", va="center", fontsize=14, weight="bold")
    ax.set_title(label)


complaint_cohort = call_outcomes.loc[call_outcomes["category"] == "COMPLAINT"].iloc[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (label, column, color) in zip(
    axes,
    [
        ("Resolved on first call (proxy)", "fcr_proxy_pct", "#2563EB"),
        ("Requires follow-up", "followup_pct", "#F59E0B"),
        ("Escalated", "escalation_pct", "#DC2626"),
    ],
    strict=True,
):
    draw_binary_donut(ax, label, complaint_cohort[column], color)
fig.suptitle("Complaint-interaction outcomes")
plt.tight_layout()
plt.show()

## 4. Language and accent experience

Accent alignment is a quality diagnostic only. It must never determine identity, authentication,
eligibility, liability, priority or an adverse decision. Differences are descriptive and may be
confounded by channel, reason, country, availability or routing.

In [ ]:
display(accent_alignment)

plt.figure(figsize=(7, 5))
plt.pie(
    accent_alignment["interactions"],
    labels=accent_alignment["accent_alignment"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"width": 0.38, "edgecolor": "white"},
)
plt.title("Accent-alignment distribution")
plt.tight_layout()
plt.show()

## 5. Transcripts, recordings and case lineage

This section measures whether the evidence needed to audit and improve a dispute journey exists.
It does not display transcript text.

In [ ]:
transcript_coverage = repo.transcript_coverage()
complaint_linkage = repo.complaint_linkage_quality()
display(transcript_coverage)
display(complaint_linkage)

In [ ]:
weights = transcript_coverage["interactions"]
weighted_metrics = {
    "Transcript available": (transcript_coverage["transcript_available_pct"] * weights).sum()
    / weights.sum(),
    "Recording available": (transcript_coverage["recording_available_pct"] * weights).sum()
    / weights.sum(),
    "Resolved on first call (proxy)": (transcript_coverage["fcr_proxy_pct"] * weights).sum()
    / weights.sum(),
}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (label, value) in zip(axes, weighted_metrics.items(), strict=True):
    draw_binary_donut(ax, label, value)
fig.suptitle("Evidence coverage and FCR proxy across interactions")
plt.tight_layout()
plt.show()

## 6. Service agents and operational sustainability

The cost model is an illustrative planning proxy, not payroll. Annual base proxies in USD are:
Colombia 7,200; Mexico 7,575; Argentina 10,200. Experience multipliers are 1.00 (Junior), 1.45
(Mid-Senior), 1.50 (Specialist) and 1.62 (Senior), followed by a 1.42 loaded-cost multiplier and 132
productive hours/month. Cost then uses each interaction's observed duration.

Sources and rationale: [Nearshore Business Solutions 2025 ranges](https://nearshorebusinesssolutions.com/call-center-representatives/)
and [HireTalent LATAM planning ranges](https://blog.hiretalent.lat/latin-america-call-center-agent-cost/).
Replace these assumptions with bank-specific fully loaded rates before financial decisions.

In [ ]:
agent_capacity = repo.agent_capacity()
interaction_costs = repo.interaction_cost_proxy()
display(agent_capacity)
display(interaction_costs)

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(interaction_costs, x="total_labor_cost_proxy_usd", y="reason_category")
plt.title("Illustrative agent labor-cost proxy by reason")
plt.xlabel("Total modeled labor cost (USD)")
plt.ylabel("Reason category")
plt.tight_layout()
plt.show()

## 7. Satisfaction surveys and digital events

The original workbench anticipated `satisfaction_surveys` and `digital_events`. They are not
present in the current `lakehouse.silver` contract, so this notebook does not fabricate metrics.
Once treated tables are published, add them to the inventory and implement aggregate adapters in
`analytics/data.py`.

In [ ]:
missing_expected_tables = sorted({"satisfaction_surveys", "digital_events"} - available_tables)
pd.DataFrame({"unavailable_table": missing_expected_tables})

## 8. Prioritized product opportunities and scorecard

1. **Create immutable lineage.** Case creation should atomically link customer, interaction,
   transcript, selected transaction, product and agent/tool actions. Monitor linkage coverage.
2. **Reduce complaint rework.** Ask one answerable question at a time, rank candidate transactions,
   confirm the selected transaction and classification, then preserve the customer statement.
   Monitor FCR proxy, follow-up, escalation and repeated complaints.
3. **Make automation safe.** LLMs can understand and summarize; deterministic/versioned rules
   authorize irreversible actions. Monitor abstention, tool success, override and rollback rates.
4. **Use humans where judgment matters.** Handoff ambiguity, denial, missing evidence and risky
   actions with a structured summary. Monitor handoff completion and context-repetition rate.
5. **Improve efficiency without hiding failures.** Track wait time, duration and cost alongside SLA,
   customer outcomes, evidence completeness and auditability.

In [ ]:
metrics = headline_metrics.iloc[0]
scorecard = pd.DataFrame(
    [
        ["Card dispute demand", "card_disputes", metrics["card_disputes"], "Trend, not target"],
        [
            "Call-center share",
            "call_center_cases/card_disputes",
            100 * metrics["call_center_cases"] / metrics["card_disputes"],
            "Context",
        ],
        ["SLA breach rate", "sla_breach_pct", metrics["sla_breach_pct"], "Lower"],
        ["Repeat complainants", "repeat_complainer_pct", metrics["repeat_complainer_pct"], "Lower"],
        [
            "Median resolution days",
            "median_resolution_days",
            metrics["median_resolution_days"],
            "Lower",
        ],
        ["P90 resolution days", "p90_resolution_days", metrics["p90_resolution_days"], "Lower"],
        ["Origin interaction linkage", "origin_link_pct", metrics["origin_link_pct"], "Higher"],
    ],
    columns=["business_question", "metric", "baseline", "desired_direction"],
)
display(scorecard)

## 9. Public delivery and AWS BI options

The accompanying Streamlit app is the recommended hackathon publication path: it reads the private
versioned DuckDB file in S3 when configured, retains MotherDuck as a compatibility fallback, caches
aggregate queries and can be hosted free on Streamlit Community Cloud. Store credentials only in the
platform's encrypted secrets, never in Git.

For an AWS-native future state:

- Amazon QuickSight/Quick can query Aurora PostgreSQL directly or query Parquet in S3 through
  Athena. It does not query a DuckDB file directly.
- Public dashboard sharing requires Enterprise/public sharing and anonymous capacity; this is
  materially more expensive than the hackathon Streamlit path.
- Official references: [supported data sources](https://docs.aws.amazon.com/quick/latest/userguide/supported-data-sources.html),
  [public dashboard sharing](https://docs.aws.amazon.com/quick/latest/userguide/share-a-dashboard-grant-access-anyone.html),
  and [pricing](https://aws.amazon.com/quicksuite/quicksight/pricing/).